# Imports

In [ ]:
from pathlib import Path

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.base import clone


# Read in Data

In [ ]:
DATA_FILE = Path("training_set_base.csv")

df = pd.read_csv(DATA_FILE, encoding="utf-8-sig").reset_index(drop=True)

ID_COLUMN = "cat_substrate"

# Split on the final underscore, e.g. KA12_A5 -> KA12 and A5.
identity_parts = (
    df[ID_COLUMN]
    .astype(str)
    .str.rsplit("_", n=1, expand=True)
)

if identity_parts.shape[1] != 2 or identity_parts.isna().any().any():
    raise ValueError(
        "Every identifier must have the form '<catalyst>_<substrate>'."
    )

meta = pd.DataFrame(
    {
        "row_index": np.arange(len(df)),
        "cat_substrate": df[ID_COLUMN],
        "catalyst": identity_parts[0].to_numpy(),
        "substrate": identity_parts[1].to_numpy(),
    }
)

display(meta.head())

# Data Set Split Functions

In [ ]:
def make_split_registry(
    meta,
    n_kfold_splits=5,
    random_state=42,
):
    """
    Construct outer-validation splits.

    All returned indices refer to row positions in the supplied meta DataFrame.
    Reset the index before calling this function.
    """
    meta = meta.reset_index(drop=True)
    n_rows = len(meta)
    dummy_X = np.zeros((n_rows, 1))

    split_registry = {
        "kfold": [],
        "LOCO": [],
        "LOSO": [],
        "double_cold_start": [],
    }

    # ---------------------------------------------------------------
    # 1. Standard row-wise K-fold
    # ---------------------------------------------------------------
    kfold = KFold(
        n_splits=n_kfold_splits,
        shuffle=True,
        random_state=random_state,
    )

    for fold_number, (train_idx, test_idx) in enumerate(
        kfold.split(dummy_X),
        start=1,
    ):
        split_registry["kfold"].append(
            {
                "regime": "kfold",
                "fold_id": f"kfold_{fold_number}",
                "train_idx": train_idx,
                "test_idx": test_idx,
                "held_out_catalysts": [],
                "held_out_substrates": [],
                "n_unused": 0,
            }
        )

    # ---------------------------------------------------------------
    # 2. Leave-one-catalyst-out
    # ---------------------------------------------------------------
    loco = LeaveOneGroupOut()

    for train_idx, test_idx in loco.split(
        dummy_X,
        groups=meta["catalyst"],
    ):
        held_out_catalysts = sorted(
            meta.iloc[test_idx]["catalyst"].unique()
        )

        split_registry["LOCO"].append(
            {
                "regime": "LOCO",
                "fold_id": (
                    "catalyst=" + ",".join(held_out_catalysts)
                ),
                "train_idx": train_idx,
                "test_idx": test_idx,
                "held_out_catalysts": held_out_catalysts,
                "held_out_substrates": [],
                "n_unused": 0,
            }
        )

    # ---------------------------------------------------------------
    # 3. Leave-one-substrate-out
    # ---------------------------------------------------------------
    loso = LeaveOneGroupOut()

    for train_idx, test_idx in loso.split(
        dummy_X,
        groups=meta["substrate"],
    ):
        held_out_substrates = sorted(
            meta.iloc[test_idx]["substrate"].unique()
        )

        split_registry["LOSO"].append(
            {
                "regime": "LOSO",
                "fold_id": (
                    "substrate=" + ",".join(held_out_substrates)
                ),
                "train_idx": train_idx,
                "test_idx": test_idx,
                "held_out_catalysts": [],
                "held_out_substrates": held_out_substrates,
                "n_unused": 0,
            }
        )

    # ---------------------------------------------------------------
    # 4. Double cold start
    # ---------------------------------------------------------------
    # Each observed catalyst-substrate pair becomes one test fold.
    # Training excludes every row containing either component.

    observed_pairs = (
        meta[["catalyst", "substrate"]]
        .drop_duplicates()
        .sort_values(["catalyst", "substrate"])
    )

    for catalyst, substrate in observed_pairs.itertuples(
        index=False,
        name=None,
    ):
        test_mask = (
            meta["catalyst"].eq(catalyst)
            & meta["substrate"].eq(substrate)
        )

        train_mask = (
            meta["catalyst"].ne(catalyst)
            & meta["substrate"].ne(substrate)
        )

        # Rows sharing exactly one held-out component are neither
        # training nor test observations in this fold.
        unused_mask = ~(train_mask | test_mask)

        train_idx = np.flatnonzero(train_mask.to_numpy())
        test_idx = np.flatnonzero(test_mask.to_numpy())

        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        split_registry["double_cold_start"].append(
            {
                "regime": "double_cold_start",
                "fold_id": (
                    f"catalyst={catalyst}|substrate={substrate}"
                ),
                "train_idx": train_idx,
                "test_idx": test_idx,
                "held_out_catalysts": [catalyst],
                "held_out_substrates": [substrate],
                "n_unused": int(unused_mask.sum()),
            }
        )

    return split_registry

In [ ]:
split_registry = make_split_registry(
    meta,
    n_kfold_splits=5,
    random_state=42,
)

In [ ]:
def audit_split_registry(split_registry, meta):
    """
    Check index separation, identity separation, and prediction coverage.
    """
    meta = meta.reset_index(drop=True)
    n_rows = len(meta)

    for regime, folds in split_registry.items():
        test_coverage = np.zeros(n_rows, dtype=int)

        for fold in folds:
            train_idx = np.asarray(fold["train_idx"])
            test_idx = np.asarray(fold["test_idx"])

            # No observation may occur in both train and test.
            assert set(train_idx).isdisjoint(test_idx)

            train_meta = meta.iloc[train_idx]
            test_meta = meta.iloc[test_idx]

            if regime == "LOCO":
                assert set(train_meta["catalyst"]).isdisjoint(
                    test_meta["catalyst"]
                )

            elif regime == "LOSO":
                assert set(train_meta["substrate"]).isdisjoint(
                    test_meta["substrate"]
                )

            elif regime == "double_cold_start":
                assert set(train_meta["catalyst"]).isdisjoint(
                    test_meta["catalyst"]
                )
                assert set(train_meta["substrate"]).isdisjoint(
                    test_meta["substrate"]
                )

            test_coverage[test_idx] += 1

        # Every row should receive exactly one outer-test prediction.
        assert np.all(test_coverage == 1), (
            f"{regime}: expected every row to be tested exactly once; "
            f"coverage values were {np.unique(test_coverage)}"
        )

        print(
            f"{regime}: audit passed "
            f"({len(folds)} folds, {test_coverage.sum()} predictions)"
        )


audit_split_registry(split_registry, meta)

In [ ]:
summary_rows = []

for regime, folds in split_registry.items():
    summary_rows.append(
        {
            "regime": regime,
            "n_folds": len(folds),
            "total_test_predictions": sum(
                len(fold["test_idx"]) for fold in folds
            ),
            "min_train_size": min(
                len(fold["train_idx"]) for fold in folds
            ),
            "max_train_size": max(
                len(fold["train_idx"]) for fold in folds
            ),
            "min_test_size": min(
                len(fold["test_idx"]) for fold in folds
            ),
            "max_test_size": max(
                len(fold["test_idx"]) for fold in folds
            ),
            "median_unused_rows": np.median(
                [fold["n_unused"] for fold in folds]
            ),
        }
    )

split_summary = pd.DataFrame(summary_rows)
display(split_summary)

In [ ]:
cv_splits = {
    regime: [
        (fold["train_idx"], fold["test_idx"])
        for fold in folds
    ]
    for regime, folds in split_registry.items()
}

kfold_splits = cv_splits["kfold"]
loco_splits = cv_splits["LOCO"]
loso_splits = cv_splits["LOSO"]
double_cold_splits = cv_splits["double_cold_start"]

In [ ]:
model_data = df.copy()

model_data["catalyst"] = meta["catalyst"].to_numpy()
model_data["substrate"] = meta["substrate"].to_numpy()

linear_feature_columns = [
    column
    for column in model_data.columns
    if re.fullmatch(r"x\d+", str(column))
]

linear_feature_columns = sorted(
    linear_feature_columns,
    key=lambda column: int(column[1:]),
)

# Based on the descriptor definitions in the SI.
catalyst_feature_columns = [
    column
    for column in linear_feature_columns
    if int(column[1:]) <= 13
]

substrate_feature_columns = [
    column
    for column in linear_feature_columns
    if int(column[1:]) >= 14
]

# Fixed catalyst–substrate interaction.
INTERACTION_COLUMN = "x24subx11_squared"

required_interaction_inputs = {"x11", "x24"}

if not required_interaction_inputs.issubset(model_data.columns):
    raise ValueError(
        "The interaction model requires x11 and x24."
    )

model_data[INTERACTION_COLUMN] = (
    model_data["x24"] - model_data["x11"]
) ** 2

interaction_feature_columns = (
    linear_feature_columns
    + [INTERACTION_COLUMN]
)

y = model_data["ddG"].astype(float).to_numpy()

print("Linear features:", linear_feature_columns)
print("Catalyst features:", catalyst_feature_columns)
print("Substrate features:", substrate_feature_columns)
print("Interaction features:", interaction_feature_columns)

# Shared inputs and utilities used by all model evaluations below.
validation_regimes = [
    "kfold",
    "LOCO",
    "LOSO",
    "double_cold_start",
]

X_categorical = model_data[["catalyst", "substrate"]].copy()


def safe_r2(y_true, y_pred):
    """Return R-squared when it is defined for the supplied fold."""
    y_true = np.asarray(y_true, dtype=float)

    if len(y_true) < 2 or np.isclose(np.var(y_true), 0.0):
        return np.nan

    return r2_score(y_true, y_pred)


def plot_oof_predictions(
    prediction_table,
    summary_table,
    regimes=validation_regimes,
    model_names=(),
):
    """Plot measured versus OOF-predicted response for each model."""
    if not model_names:
        raise ValueError("At least one model name must be supplied.")

    figure, axes = plt.subplots(
        len(regimes),
        len(model_names),
        figsize=(4 * len(model_names), 4 * len(regimes)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    plot_min = min(
        prediction_table["y_true"].min(),
        prediction_table["y_pred"].min(),
    )
    plot_max = max(
        prediction_table["y_true"].max(),
        prediction_table["y_pred"].max(),
    )
    padding = 0.05 * (plot_max - plot_min)
    limits = (plot_min - padding, plot_max + padding)

    for row, regime in enumerate(regimes):
        for column, model_name in enumerate(model_names):
            axis = axes[row, column]
            subset = prediction_table.loc[
                prediction_table["regime"].eq(regime)
                & prediction_table["model"].eq(model_name)
            ]
            metrics = summary_table.loc[
                summary_table["regime"].eq(regime)
                & summary_table["model"].eq(model_name)
            ].iloc[0]

            axis.scatter(
                subset["y_true"],
                subset["y_pred"],
                alpha=0.75,
                s=28,
                edgecolor="none",
            )
            axis.plot(limits, limits, "--", color="0.35", linewidth=1)
            axis.set_xlim(limits)
            axis.set_ylim(limits)
            axis.set_aspect("equal", adjustable="box")
            axis.text(
                0.04,
                0.96,
                f"R² = {metrics['r2']:.2f}\nMAE = {metrics['mae']:.2f}",
                transform=axis.transAxes,
                va="top",
            )

            if row == 0:
                axis.set_title(model_name.replace("_", " "))
            if column == 0:
                axis.set_ylabel(f"{regime}\nOOF prediction")
            if row == len(regimes) - 1:
                axis.set_xlabel("Measured ΔΔG")

    figure.tight_layout()
    return figure, axes

## One-hot-encoded Ridge baseline

This categorical model contains separate one-hot indicators for catalyst and substrate identity. Ridge regularization is selected independently within every outer-training fold using repeated stratified inner cross-validation. The encoder is part of the pipeline, so it is fitted again inside every inner split; an unseen identity is handled by an all-zero indicator block.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedKFold,
    RepeatedStratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


RIDGE_ALPHAS = np.logspace(-4, 4, 25)
INNER_N_SPLITS = 5
INNER_N_REPEATS = 4
INNER_RANDOM_STATE = 42

categorical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "identity",
            OneHotEncoder(handle_unknown="ignore"),
            ["catalyst", "substrate"],
        )
    ],
    remainder="drop",
)

categorical_ridge_pipeline = Pipeline(
    steps=[
        ("one_hot", categorical_preprocessor),
        ("ridge", Ridge(solver="lsqr")),
    ]
)

In [ ]:
def make_repeated_regression_cv(
    X,
    y,
    n_splits=INNER_N_SPLITS,
    n_repeats=INNER_N_REPEATS,
    random_state=INNER_RANDOM_STATE,
    n_strata=5,
):
    """Create repeated CV splits, stratifying response quantiles when possible."""
    y = np.asarray(y, dtype=float)
    y_binned = pd.qcut(
        pd.Series(y),
        q=min(n_strata, len(y)),
        labels=False,
        duplicates="drop",
    )

    bin_counts = y_binned.value_counts()
    can_stratify = (
        y_binned.nunique() >= 2
        and len(bin_counts) > 0
        and bin_counts.min() >= 2
    )

    if can_stratify:
        effective_splits = min(n_splits, int(bin_counts.min()))
        cv = RepeatedStratifiedKFold(
            n_splits=effective_splits,
            n_repeats=n_repeats,
            random_state=random_state,
        )
        splits = list(cv.split(X, y_binned))
        method = "RepeatedStratifiedKFold"
    else:
        effective_splits = min(n_splits, len(y))
        if effective_splits < 2:
            raise ValueError("At least two training observations are required")
        cv = RepeatedKFold(
            n_splits=effective_splits,
            n_repeats=n_repeats,
            random_state=random_state,
        )
        splits = list(cv.split(X, y))
        method = "RepeatedKFold"

    return splits, {
        "inner_cv_method": method,
        "inner_n_splits": effective_splits,
        "inner_n_repeats": n_repeats,
        "inner_n_strata": int(y_binned.nunique()),
    }

In [ ]:
def evaluate_categorical_ridge(
    regime,
    outer_folds,
    X_model,
    y,
    meta,
    estimator=categorical_ridge_pipeline,
    alphas=RIDGE_ALPHAS,
):
    """Nested-CV evaluation of the catalyst/substrate Ridge model."""
    model_name = "categorical_ridge"
    X_model = X_model.reset_index(drop=True)
    meta = meta.reset_index(drop=True)
    y = np.asarray(y, dtype=float)

    oof_prediction = np.full(len(y), np.nan, dtype=float)
    test_coverage = np.zeros(len(y), dtype=int)
    fold_rows = []
    fitted_searches = {}

    for fold in outer_folds:
        train_idx = np.asarray(fold["train_idx"], dtype=int)
        test_idx = np.asarray(fold["test_idx"], dtype=int)
        X_train = X_model.iloc[train_idx]
        X_test = X_model.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        inner_splits, inner_details = make_repeated_regression_cv(
            X_train,
            y_train,
        )

        search = GridSearchCV(
            estimator=clone(estimator),
            param_grid={"ridge__alpha": np.asarray(alphas)},
            scoring="neg_mean_absolute_error",
            cv=inner_splits,
            refit=True,
            n_jobs=-1,
            return_train_score=False,
        )
        search.fit(X_train, y_train)
        fold_prediction = np.asarray(
            search.predict(X_test),
            dtype=float,
        )

        if not np.all(np.isfinite(fold_prediction)):
            raise ValueError(
                f"{regime}, {fold['fold_id']}: non-finite prediction"
            )

        if np.any(test_coverage[test_idx] != 0):
            raise ValueError(
                f"{regime}: an observation occurs in more than one test fold"
            )

        oof_prediction[test_idx] = fold_prediction
        test_coverage[test_idx] += 1
        fitted_searches[fold["fold_id"]] = search

        fold_rows.append(
            {
                "model": model_name,
                "regime": regime,
                "fold_id": fold["fold_id"],
                "held_out_catalysts": ";".join(
                    fold["held_out_catalysts"]
                ),
                "held_out_substrates": ";".join(
                    fold["held_out_substrates"]
                ),
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_unused": fold["n_unused"],
                "best_alpha": search.best_params_["ridge__alpha"],
                "best_inner_mae": -search.best_score_,
                "r2": safe_r2(y_test, fold_prediction),
                "mae": mean_absolute_error(y_test, fold_prediction),
                "rmse": np.sqrt(
                    mean_squared_error(y_test, fold_prediction)
                ),
                **inner_details,
            }
        )

    if not np.all(test_coverage == 1):
        raise ValueError(
            f"{regime}: expected one OOF prediction per observation; "
            f"coverage values were {np.unique(test_coverage)}"
        )

    prediction_table = meta.copy()
    prediction_table["model"] = model_name
    prediction_table["regime"] = regime
    prediction_table["y_true"] = y
    prediction_table["y_pred"] = oof_prediction
    prediction_table["residual"] = y - oof_prediction
    prediction_table["absolute_error"] = np.abs(
        prediction_table["residual"]
    )

    summary = {
        "model": model_name,
        "regime": regime,
        "n_observations": len(y),
        "n_outer_folds": len(outer_folds),
        "r2": safe_r2(y, oof_prediction),
        "mae": mean_absolute_error(y, oof_prediction),
        "rmse": np.sqrt(mean_squared_error(y, oof_prediction)),
        "median_absolute_error": np.median(
            np.abs(y - oof_prediction)
        ),
    }

    return {
        "summary": summary,
        "predictions": prediction_table,
        "fold_metrics": pd.DataFrame(fold_rows),
        "fitted_searches": fitted_searches,
    }

In [ ]:
categorical_ridge_results = {}

for regime in validation_regimes:
    print(f"Evaluating categorical Ridge: {regime}")
    categorical_ridge_results[regime] = evaluate_categorical_ridge(
        regime=regime,
        outer_folds=split_registry[regime],
        X_model=X_categorical,
        y=y,
        meta=meta,
    )

ridge_summary = pd.DataFrame(
    [result["summary"] for result in categorical_ridge_results.values()]
).sort_values("regime").reset_index(drop=True)

ridge_predictions = pd.concat(
    [
        result["predictions"]
        for result in categorical_ridge_results.values()
    ],
    ignore_index=True,
)

ridge_fold_metrics = pd.concat(
    [
        result["fold_metrics"]
        for result in categorical_ridge_results.values()
    ],
    ignore_index=True,
)

display(
    ridge_summary.style.format(
        {
            "r2": "{:.3f}",
            "mae": "{:.3f}",
            "rmse": "{:.3f}",
            "median_absolute_error": "{:.3f}",
        }
    )
)

display(
    ridge_fold_metrics.groupby("regime")["best_alpha"]
    .describe()[["count", "min", "50%", "max"]]
)

In [ ]:
ridge_prediction_figure, ridge_prediction_axes = (
    plot_oof_predictions(
        prediction_table=ridge_predictions,
        summary_table=ridge_summary,
        regimes=validation_regimes,
        model_names=("categorical_ridge",),
    )
)

RIDGE_RESULTS_DIR = Path("results/categorical_ridge")
RIDGE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ridge_summary.to_csv(
    RIDGE_RESULTS_DIR / "categorical_ridge_summary.csv",
    index=False,
)
ridge_predictions.to_csv(
    RIDGE_RESULTS_DIR / "categorical_ridge_predictions.csv",
    index=False,
)
ridge_fold_metrics.to_csv(
    RIDGE_RESULTS_DIR / "categorical_ridge_fold_metrics.csv",
    index=False,
)
ridge_prediction_figure.savefig(
    RIDGE_RESULTS_DIR / "categorical_ridge_oof_predictions.png",
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved categorical-Ridge results to {RIDGE_RESULTS_DIR.resolve()}")

## Descriptor-space k-nearest-neighbors baseline

This similarity baseline uses the fixed catalyst and substrate descriptor vector. `StandardScaler` is fitted inside the pipeline so that distances are calculated between standardized features without preprocessing leakage. The number of neighbors, neighbor weighting, and Manhattan versus Euclidean distance are selected using the same repeated stratified inner-CV helper and random seed as the categorical Ridge model.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler


X_knn = model_data[linear_feature_columns].astype(float).copy()

if not np.all(np.isfinite(X_knn.to_numpy())):
    raise ValueError("The k-NN descriptor matrix contains missing values")

knn_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "knn",
            KNeighborsRegressor(
                metric="minkowski",
                algorithm="brute",
                n_jobs=1,
            ),
        ),
    ]
)

KNN_PARAM_GRID = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

print("Raw k-NN descriptor shape:", X_knn.shape)
print("k-NN descriptor columns:", list(X_knn.columns))

In [ ]:
def evaluate_nested_pipeline(
    model_name,
    estimator,
    param_grid,
    regime,
    outer_folds,
    X_model,
    y,
    meta,
):
    """Evaluate a tuned pipeline using complete outer-fold predictions."""
    X_model = X_model.reset_index(drop=True)
    meta = meta.reset_index(drop=True)
    y = np.asarray(y, dtype=float)

    oof_prediction = np.full(len(y), np.nan, dtype=float)
    test_coverage = np.zeros(len(y), dtype=int)
    fold_rows = []
    fitted_searches = {}

    for fold in outer_folds:
        train_idx = np.asarray(fold["train_idx"], dtype=int)
        test_idx = np.asarray(fold["test_idx"], dtype=int)
        X_train = X_model.iloc[train_idx]
        X_test = X_model.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        # This is the same deterministic inner-split generator used
        # for categorical Ridge. X values are not used to make splits.
        inner_splits, inner_details = make_repeated_regression_cv(
            X_train,
            y_train,
        )

        search = GridSearchCV(
            estimator=clone(estimator),
            param_grid=param_grid,
            scoring="neg_mean_absolute_error",
            cv=inner_splits,
            refit=True,
            n_jobs=-1,
            return_train_score=False,
            error_score="raise",
        )
        search.fit(X_train, y_train)
        fold_prediction = np.asarray(
            search.predict(X_test),
            dtype=float,
        )

        if not np.all(np.isfinite(fold_prediction)):
            raise ValueError(
                f"{regime}, {fold['fold_id']}: non-finite prediction"
            )

        if np.any(test_coverage[test_idx] != 0):
            raise ValueError(
                f"{regime}: an observation occurs in more than one test fold"
            )

        oof_prediction[test_idx] = fold_prediction
        test_coverage[test_idx] += 1
        fitted_searches[fold["fold_id"]] = search

        best_parameters = {
            f"best_{name.split('__')[-1]}": value
            for name, value in search.best_params_.items()
        }

        fold_rows.append(
            {
                "model": model_name,
                "regime": regime,
                "fold_id": fold["fold_id"],
                "held_out_catalysts": ";".join(
                    fold["held_out_catalysts"]
                ),
                "held_out_substrates": ";".join(
                    fold["held_out_substrates"]
                ),
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_unused": fold["n_unused"],
                "best_inner_mae": -search.best_score_,
                "r2": safe_r2(y_test, fold_prediction),
                "mae": mean_absolute_error(y_test, fold_prediction),
                "rmse": np.sqrt(
                    mean_squared_error(y_test, fold_prediction)
                ),
                **best_parameters,
                **inner_details,
            }
        )

    if not np.all(test_coverage == 1):
        raise ValueError(
            f"{regime}: expected one OOF prediction per observation; "
            f"coverage values were {np.unique(test_coverage)}"
        )

    prediction_table = meta.copy()
    prediction_table["model"] = model_name
    prediction_table["regime"] = regime
    prediction_table["y_true"] = y
    prediction_table["y_pred"] = oof_prediction
    prediction_table["residual"] = y - oof_prediction
    prediction_table["absolute_error"] = np.abs(
        prediction_table["residual"]
    )

    summary = {
        "model": model_name,
        "regime": regime,
        "n_observations": len(y),
        "n_outer_folds": len(outer_folds),
        "r2": safe_r2(y, oof_prediction),
        "mae": mean_absolute_error(y, oof_prediction),
        "rmse": np.sqrt(mean_squared_error(y, oof_prediction)),
        "median_absolute_error": np.median(
            np.abs(y - oof_prediction)
        ),
    }

    return {
        "summary": summary,
        "predictions": prediction_table,
        "fold_metrics": pd.DataFrame(fold_rows),
        "fitted_searches": fitted_searches,
    }

In [ ]:
knn_results = {}

for regime in validation_regimes:
    print(f"Evaluating descriptor k-NN: {regime}")
    knn_results[regime] = evaluate_nested_pipeline(
        model_name="descriptor_knn",
        estimator=knn_pipeline,
        param_grid=KNN_PARAM_GRID,
        regime=regime,
        outer_folds=split_registry[regime],
        X_model=X_knn,
        y=y,
        meta=meta,
    )

knn_summary = pd.DataFrame(
    [result["summary"] for result in knn_results.values()]
).sort_values("regime").reset_index(drop=True)

knn_predictions = pd.concat(
    [result["predictions"] for result in knn_results.values()],
    ignore_index=True,
)

knn_fold_metrics = pd.concat(
    [result["fold_metrics"] for result in knn_results.values()],
    ignore_index=True,
)

display(
    knn_summary.style.format(
        {
            "r2": "{:.3f}",
            "mae": "{:.3f}",
            "rmse": "{:.3f}",
            "median_absolute_error": "{:.3f}",
        }
    )
)

In [ ]:
knn_hyperparameter_counts = (
    knn_fold_metrics.groupby(
        ["regime", "best_n_neighbors", "best_weights", "best_p"]
    )
    .size()
    .rename("n_outer_folds")
    .reset_index()
    .sort_values(["regime", "n_outer_folds"], ascending=[True, False])
)

display(knn_hyperparameter_counts)

baseline_summary = pd.concat(
    [ridge_summary, knn_summary],
    ignore_index=True,
)

for metric in ["mae", "rmse", "r2"]:
    comparison = baseline_summary.pivot(
        index="regime",
        columns="model",
        values=metric,
    ).reindex(validation_regimes)

    print(metric.upper())
    display(comparison.style.format("{:.3f}"))

In [ ]:
knn_prediction_figure, knn_prediction_axes = plot_oof_predictions(
    prediction_table=knn_predictions,
    summary_table=knn_summary,
    regimes=validation_regimes,
    model_names=("descriptor_knn",),
)

KNN_RESULTS_DIR = Path("results/knn_similarity")
KNN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

knn_summary.to_csv(
    KNN_RESULTS_DIR / "knn_summary.csv",
    index=False,
)
knn_predictions.to_csv(
    KNN_RESULTS_DIR / "knn_predictions.csv",
    index=False,
)
knn_fold_metrics.to_csv(
    KNN_RESULTS_DIR / "knn_fold_metrics.csv",
    index=False,
)
knn_hyperparameter_counts.to_csv(
    KNN_RESULTS_DIR / "knn_hyperparameter_counts.csv",
    index=False,
)
baseline_summary.to_csv(
    KNN_RESULTS_DIR / "all_baseline_summary.csv",
    index=False,
)
knn_prediction_figure.savefig(
    KNN_RESULTS_DIR / "knn_oof_predictions.png",
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved k-NN results to {KNN_RESULTS_DIR.resolve()}")

## Linear- and interaction-feature LASSO models

Both LASSO models use the same outer folds and the same repeated stratified inner-CV generator as categorical Ridge and descriptor k-NN. `StandardScaler` remains inside the pipeline and is therefore fitted only on each inner-training partition. Following the original LASSO workflow, alpha is selected with the one-standard-error rule applied to inner-validation MSE. The interaction model adds only the fixed, prespecified `(x24 - x11)²` term to the linear descriptor set.

In [ ]:
from sklearn.linear_model import Lasso


# Match the alpha grid used in the original LASSO notebook.
LASSO_ALPHAS = np.logspace(-4, -1, 100)

lasso_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "lasso",
            Lasso(
                max_iter=1000000,
                precompute=True,
                random_state=42,
                selection="cyclic",
            ),
        ),
    ]
)

LASSO_PARAM_GRID = {
    "lasso__alpha": LASSO_ALPHAS,
}

X_lasso_linear = (
    model_data[linear_feature_columns]
    .astype(float)
    .copy()
)
X_lasso_interaction = (
    model_data[interaction_feature_columns]
    .astype(float)
    .copy()
)

for name, matrix in {
    "linear_lasso": X_lasso_linear,
    "interaction_lasso": X_lasso_interaction,
}.items():
    if not np.all(np.isfinite(matrix.to_numpy())):
        raise ValueError(f"{name} contains missing or infinite values")

LASSO_MODEL_SPECS = {
    "linear_lasso": {
        "X": X_lasso_linear,
        "feature_names": linear_feature_columns,
    },
    "interaction_lasso": {
        "X": X_lasso_interaction,
        "feature_names": interaction_feature_columns,
    },
}

print("Linear-LASSO shape:", X_lasso_linear.shape)
print("Interaction-LASSO shape:", X_lasso_interaction.shape)

In [ ]:
def make_one_se_refit(n_inner_splits):
    """Return a GridSearchCV rule selecting the largest 1-SE alpha."""
    def select_one_se_index(cv_results):
        mean_mse = -np.asarray(cv_results["mean_test_score"])
        std_mse = np.asarray(cv_results["std_test_score"])
        alphas = np.asarray(
            [params["lasso__alpha"] for params in cv_results["params"]]
        )
        index_min = int(np.argmin(mean_mse))
        threshold = (
            mean_mse[index_min]
            + std_mse[index_min] / np.sqrt(n_inner_splits)
        )
        eligible = np.flatnonzero(mean_mse <= threshold)
        return int(eligible[np.argmax(alphas[eligible])])

    return select_one_se_index


def evaluate_lasso_one_se(
    model_name,
    regime,
    outer_folds,
    X_model,
    y,
    meta,
    estimator=lasso_pipeline,
    param_grid=LASSO_PARAM_GRID,
):
    """Nested LASSO evaluation with MSE-based one-SE selection."""
    X_model = X_model.reset_index(drop=True)
    meta = meta.reset_index(drop=True)
    y = np.asarray(y, dtype=float)
    oof_prediction = np.full(len(y), np.nan, dtype=float)
    test_coverage = np.zeros(len(y), dtype=int)
    fold_rows = []

    for fold in outer_folds:
        train_idx = np.asarray(fold["train_idx"], dtype=int)
        test_idx = np.asarray(fold["test_idx"], dtype=int)
        X_train = X_model.iloc[train_idx]
        X_test = X_model.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        inner_splits, inner_details = make_repeated_regression_cv(
            X_train,
            y_train,
        )
        n_inner_splits = len(inner_splits)
        search = GridSearchCV(
            estimator=clone(estimator),
            param_grid=param_grid,
            scoring="neg_mean_squared_error",
            cv=inner_splits,
            refit=make_one_se_refit(n_inner_splits),
            n_jobs=-1,
            return_train_score=False,
            error_score="raise",
        )
        search.fit(X_train, y_train)
        fold_prediction = np.asarray(search.predict(X_test), dtype=float)

        cv_results = search.cv_results_
        mean_mse = -np.asarray(cv_results["mean_test_score"])
        std_mse = np.asarray(cv_results["std_test_score"])
        candidate_alphas = np.asarray(
            [params["lasso__alpha"] for params in cv_results["params"]]
        )
        index_min = int(np.argmin(mean_mse))
        alpha_min = float(candidate_alphas[index_min])
        alpha_1se = float(search.best_params_["lasso__alpha"])
        se_at_min = float(
            std_mse[index_min] / np.sqrt(n_inner_splits)
        )

        if not np.all(np.isfinite(fold_prediction)):
            raise ValueError(
                f"{regime}, {fold['fold_id']}: non-finite prediction"
            )
        if np.any(test_coverage[test_idx] != 0):
            raise ValueError(
                f"{regime}: an observation occurs in more than one test fold"
            )

        oof_prediction[test_idx] = fold_prediction
        test_coverage[test_idx] += 1
        fold_rows.append(
            {
                "model": model_name,
                "regime": regime,
                "fold_id": fold["fold_id"],
                "held_out_catalysts": ";".join(
                    fold["held_out_catalysts"]
                ),
                "held_out_substrates": ";".join(
                    fold["held_out_substrates"]
                ),
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_unused": fold["n_unused"],
                "alpha_min": alpha_min,
                "alpha_1se": alpha_1se,
                "alpha_selected": alpha_1se,
                "alpha_choice": "1se",
                "mean_inner_mse_min": float(mean_mse[index_min]),
                "se_inner_mse_at_min": se_at_min,
                "one_se_mse_threshold": float(
                    mean_mse[index_min] + se_at_min
                ),
                "r2": safe_r2(y_test, fold_prediction),
                "mae": mean_absolute_error(y_test, fold_prediction),
                "rmse": np.sqrt(
                    mean_squared_error(y_test, fold_prediction)
                ),
                **inner_details,
            }
        )

    if not np.all(test_coverage == 1):
        raise ValueError(
            f"{regime}: expected one OOF prediction per observation; "
            f"coverage values were {np.unique(test_coverage)}"
        )

    prediction_table = meta.copy()
    prediction_table["model"] = model_name
    prediction_table["regime"] = regime
    prediction_table["y_true"] = y
    prediction_table["y_pred"] = oof_prediction
    prediction_table["residual"] = y - oof_prediction
    prediction_table["absolute_error"] = np.abs(
        prediction_table["residual"]
    )
    summary = {
        "model": model_name,
        "regime": regime,
        "n_observations": len(y),
        "n_outer_folds": len(outer_folds),
        "alpha_choice": "1se",
        "r2": safe_r2(y, oof_prediction),
        "mae": mean_absolute_error(y, oof_prediction),
        "rmse": np.sqrt(mean_squared_error(y, oof_prediction)),
        "median_absolute_error": np.median(
            np.abs(y - oof_prediction)
        ),
    }
    return {
        "summary": summary,
        "predictions": prediction_table,
        "fold_metrics": pd.DataFrame(fold_rows),
    }


lasso_results = {}

for model_name, specification in LASSO_MODEL_SPECS.items():
    for regime in validation_regimes:
        print(f"Evaluating {model_name} with 1-SE alpha: {regime}")
        lasso_results[(regime, model_name)] = evaluate_lasso_one_se(
            model_name=model_name,
            regime=regime,
            outer_folds=split_registry[regime],
            X_model=specification["X"],
            y=y,
            meta=meta,
        )

lasso_summary = pd.DataFrame(
    [result["summary"] for result in lasso_results.values()]
).sort_values(["regime", "model"]).reset_index(drop=True)

lasso_predictions = pd.concat(
    [result["predictions"] for result in lasso_results.values()],
    ignore_index=True,
)

lasso_fold_metrics = pd.concat(
    [result["fold_metrics"] for result in lasso_results.values()],
    ignore_index=True,
)

display(
    lasso_summary.style.format(
        {
            "r2": "{:.3f}",
            "mae": "{:.3f}",
            "rmse": "{:.3f}",
            "median_absolute_error": "{:.3f}",
        }
    )
)

In [ ]:
lasso_fold_metrics["alpha_1se_to_min_ratio"] = (
    lasso_fold_metrics["alpha_1se"]
    / lasso_fold_metrics["alpha_min"]
)

lasso_hyperparameter_summary = (
    lasso_fold_metrics.groupby(["model", "regime"], as_index=False)
    .agg(
        n_outer_folds=("fold_id", "nunique"),
        alpha_min_cv_median=("alpha_min", "median"),
        alpha_1se_min=("alpha_1se", "min"),
        alpha_1se_median=("alpha_1se", "median"),
        alpha_1se_max=("alpha_1se", "max"),
        median_alpha_1se_to_min_ratio=(
            "alpha_1se_to_min_ratio",
            "median",
        ),
    )
)

display(
    lasso_hyperparameter_summary.style.format(
        {
            "alpha_min_cv_median": "{:.5g}",
            "alpha_1se_min": "{:.5g}",
            "alpha_1se_median": "{:.5g}",
            "alpha_1se_max": "{:.5g}",
            "median_alpha_1se_to_min_ratio": "{:.2f}",
        }
    )
)

In [ ]:
all_model_summary = pd.concat(
    [baseline_summary, lasso_summary],
    ignore_index=True,
)

for metric in ["mae", "rmse", "r2"]:
    comparison = all_model_summary.pivot(
        index="regime",
        columns="model",
        values=metric,
    ).reindex(validation_regimes)

    print(metric.upper())
    display(comparison.style.format("{:.3f}"))

LASSO_HIGHLIGHT_COLORS = {
    "A5": "#E69F00",
    "A6": "#0072B2",
}


def plot_lasso_predictions(
    prediction_table,
    summary_table,
    fold_metrics,
    regimes=validation_regimes,
    model_names=("linear_lasso", "interaction_lasso"),
    highlight_colors=LASSO_HIGHLIGHT_COLORS,
):
    """Plot LASSO OOF predictions and fold-level alpha summaries."""
    figure, axes = plt.subplots(
        len(regimes),
        len(model_names),
        figsize=(4 * len(model_names), 4 * len(regimes)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    plot_min = min(
        prediction_table["y_true"].min(),
        prediction_table["y_pred"].min(),
    )
    plot_max = max(
        prediction_table["y_true"].max(),
        prediction_table["y_pred"].max(),
    )
    padding = 0.05 * (plot_max - plot_min)
    limits = (plot_min - padding, plot_max + padding)

    for row, regime in enumerate(regimes):
        for column, model_name in enumerate(model_names):
            axis = axes[row, column]
            subset = prediction_table.loc[
                prediction_table["regime"].eq(regime)
                & prediction_table["model"].eq(model_name)
            ]
            metrics = summary_table.loc[
                summary_table["regime"].eq(regime)
                & summary_table["model"].eq(model_name)
            ].iloc[0]
            median_alpha = fold_metrics.loc[
                fold_metrics["regime"].eq(regime)
                & fold_metrics["model"].eq(model_name),
                "alpha_selected",
            ].median()

            axis.scatter(
                subset["y_true"],
                subset["y_pred"],
                color="0.72",
                alpha=0.60,
                s=26,
                edgecolor="none",
                label="Other substrates",
                zorder=1,
            )

            for substrate, color in highlight_colors.items():
                highlighted = subset.loc[
                    subset["substrate"].eq(substrate)
                ]
                axis.scatter(
                    highlighted["y_true"],
                    highlighted["y_pred"],
                    color=color,
                    alpha=0.90,
                    s=38,
                    edgecolor="white",
                    linewidth=0.45,
                    label=substrate,
                    zorder=2,
                )

            axis.plot(limits, limits, "--", color="0.35", linewidth=1)
            axis.set_xlim(limits)
            axis.set_ylim(limits)
            axis.set_aspect("equal", adjustable="box")
            axis.text(
                0.04,
                0.96,
                (
                    f"R² = {metrics['r2']:.2f}\n"
                    f"MAE = {metrics['mae']:.2f}\n"
                    f"median α₁SE = {median_alpha:.5f}"
                ),
                transform=axis.transAxes,
                va="top",
                bbox={
                    "facecolor": "white",
                    "edgecolor": "none",
                    "alpha": 0.75,
                },
            )

            if row == 0:
                axis.set_title(model_name.replace("_", " "))
            if column == 0:
                axis.set_ylabel(f"{regime}\nOOF prediction")
            if row == len(regimes) - 1:
                axis.set_xlabel("Measured ΔΔG")

    legend_handles, legend_labels = axes[0, 0].get_legend_handles_labels()
    figure.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=len(legend_labels),
        frameon=False,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.98))
    return figure, axes


lasso_prediction_figure, lasso_prediction_axes = plot_lasso_predictions(
    prediction_table=lasso_predictions,
    summary_table=lasso_summary,
    fold_metrics=lasso_fold_metrics,
)


COMPARISON_MODELS = (
    "categorical_ridge",
    "descriptor_knn",
    "linear_lasso",
    "interaction_lasso",
)

MODEL_DISPLAY_NAMES = {
    "categorical_ridge": "Categorical Ridge",
    "descriptor_knn": "Descriptor k-NN",
    "linear_lasso": "Linear LASSO",
    "interaction_lasso": "Interaction LASSO",
}

model_comparison_summary = pd.concat(
    [ridge_summary, knn_summary, lasso_summary],
    ignore_index=True,
)
model_comparison_predictions = pd.concat(
    [ridge_predictions, knn_predictions, lasso_predictions],
    ignore_index=True,
)


def model_hyperparameter_note(model_name, regime):
    """Return a compact fold-level hyperparameter summary."""
    if model_name == "categorical_ridge":
        median_alpha = ridge_fold_metrics.loc[
            ridge_fold_metrics["regime"].eq(regime),
            "best_alpha",
        ].median()
        return f"median α = {median_alpha:.5g}"

    if model_name in {"linear_lasso", "interaction_lasso"}:
        median_alpha = lasso_fold_metrics.loc[
            lasso_fold_metrics["regime"].eq(regime)
            & lasso_fold_metrics["model"].eq(model_name),
            "alpha_selected",
        ].median()
        return f"median α₁SE = {median_alpha:.5f}"

    if model_name == "interaction_elastic_net":
        selected = elastic_net_fold_metrics.loc[
            elastic_net_fold_metrics["regime"].eq(regime)
        ]
        median_alpha = selected["alpha_selected"].median()
        modal_l1_ratio = selected["l1_ratio_selected"].mode().iloc[0]
        return (
            f"median α₁SE = {median_alpha:.5f}\n"
            f"mode l1 ratio = {modal_l1_ratio:.2f}"
        )

    return None


def plot_model_comparison(
    prediction_table,
    summary_table,
    regimes=validation_regimes,
    model_names=COMPARISON_MODELS,
    highlight_colors=None,
):
    """Plot the four model classes on identical OOF axes."""
    figure, axes = plt.subplots(
        len(regimes),
        len(model_names),
        figsize=(3.6 * len(model_names), 3.6 * len(regimes)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    plot_min = min(
        prediction_table["y_true"].min(),
        prediction_table["y_pred"].min(),
    )
    plot_max = max(
        prediction_table["y_true"].max(),
        prediction_table["y_pred"].max(),
    )
    padding = 0.05 * (plot_max - plot_min)
    limits = (plot_min - padding, plot_max + padding)

    for row, regime in enumerate(regimes):
        for column, model_name in enumerate(model_names):
            axis = axes[row, column]
            subset = prediction_table.loc[
                prediction_table["regime"].eq(regime)
                & prediction_table["model"].eq(model_name)
            ]
            metrics = summary_table.loc[
                summary_table["regime"].eq(regime)
                & summary_table["model"].eq(model_name)
            ].iloc[0]

            if highlight_colors is None:
                axis.scatter(
                    subset["y_true"],
                    subset["y_pred"],
                    color="0.55",
                    alpha=0.70,
                    s=24,
                    edgecolor="none",
                    zorder=1,
                )
            else:
                is_highlighted = subset["substrate"].isin(
                    highlight_colors
                )
                background = subset.loc[~is_highlighted]
                axis.scatter(
                    background["y_true"],
                    background["y_pred"],
                    color="0.72",
                    alpha=0.60,
                    s=24,
                    edgecolor="none",
                    label="Other substrates",
                    zorder=1,
                )

                for substrate, color in highlight_colors.items():
                    highlighted = subset.loc[
                        subset["substrate"].eq(substrate)
                    ]
                    axis.scatter(
                        highlighted["y_true"],
                        highlighted["y_pred"],
                        color=color,
                        alpha=0.90,
                        s=36,
                        edgecolor="white",
                        linewidth=0.45,
                        label=substrate,
                        zorder=2,
                    )

            axis.plot(limits, limits, "--", color="0.35", linewidth=1)
            axis.set_xlim(limits)
            axis.set_ylim(limits)
            axis.set_aspect("equal", adjustable="box")

            annotation_lines = [
                f"R² = {metrics['r2']:.2f}",
                f"MAE = {metrics['mae']:.2f}",
            ]
            hyperparameter_note = model_hyperparameter_note(
                model_name,
                regime,
            )
            if hyperparameter_note is not None:
                annotation_lines.append(hyperparameter_note)

            axis.text(
                0.04,
                0.96,
                "\n".join(annotation_lines),
                transform=axis.transAxes,
                va="top",
                bbox={
                    "facecolor": "white",
                    "edgecolor": "none",
                    "alpha": 0.75,
                },
            )

            if row == 0:
                axis.set_title(MODEL_DISPLAY_NAMES[model_name])
            if column == 0:
                axis.set_ylabel(f"{regime}\nOOF prediction")
            if row == len(regimes) - 1:
                axis.set_xlabel("Measured ΔΔG")

    top_margin = 0.98
    if highlight_colors is not None:
        legend_handles, legend_labels = (
            axes[0, 0].get_legend_handles_labels()
        )
        figure.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.0),
            ncol=len(legend_labels),
            frameon=False,
        )
        top_margin = 0.965

    figure.tight_layout(rect=(0, 0, 1, top_margin))
    return figure, axes


model_comparison_neutral_figure, model_comparison_neutral_axes = (
    plot_model_comparison(
        prediction_table=model_comparison_predictions,
        summary_table=model_comparison_summary,
    )
)

model_comparison_highlighted_figure, model_comparison_highlighted_axes = (
    plot_model_comparison(
        prediction_table=model_comparison_predictions,
        summary_table=model_comparison_summary,
        highlight_colors=LASSO_HIGHLIGHT_COLORS,
    )
)

In [ ]:
LASSO_RESULTS_DIR = Path("results/lasso_models")
LASSO_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

lasso_summary.to_csv(
    LASSO_RESULTS_DIR / "lasso_summary.csv",
    index=False,
)
lasso_predictions.to_csv(
    LASSO_RESULTS_DIR / "lasso_predictions.csv",
    index=False,
)
lasso_fold_metrics.to_csv(
    LASSO_RESULTS_DIR / "lasso_fold_metrics.csv",
    index=False,
)
lasso_hyperparameter_summary.to_csv(
    LASSO_RESULTS_DIR / "lasso_hyperparameter_summary.csv",
    index=False,
)
all_model_summary.to_csv(
    LASSO_RESULTS_DIR / "all_model_summary.csv",
    index=False,
)
lasso_prediction_figure.savefig(
    LASSO_RESULTS_DIR / "lasso_oof_predictions.png",
    dpi=300,
    bbox_inches="tight",
)

model_comparison_summary.to_csv(
    LASSO_RESULTS_DIR / "model_comparison_summary.csv",
    index=False,
)
model_comparison_predictions.to_csv(
    LASSO_RESULTS_DIR / "model_comparison_predictions.csv",
    index=False,
)
model_comparison_neutral_figure.savefig(
    LASSO_RESULTS_DIR / "model_comparison_oof_neutral.png",
    dpi=300,
    bbox_inches="tight",
)
model_comparison_highlighted_figure.savefig(
    LASSO_RESULTS_DIR / "model_comparison_oof_A5_A6.png",
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved LASSO results to {LASSO_RESULTS_DIR.resolve()}")

## Interaction-feature Elastic Net comparison

In [ ]:
from sklearn.linear_model import ElasticNet


ELASTIC_NET_ALPHAS = LASSO_ALPHAS.copy()
ELASTIC_NET_L1_RATIOS = np.array([0.50, 0.80, 0.95])

elastic_net_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "elastic_net",
            ElasticNet(
                max_iter=1000000,
                precompute=True,
                random_state=42,
                selection="cyclic",
            ),
        ),
    ]
)

ELASTIC_NET_PARAM_GRID = {
    "elastic_net__alpha": ELASTIC_NET_ALPHAS,
    "elastic_net__l1_ratio": ELASTIC_NET_L1_RATIOS,
}


def make_elastic_net_one_se_refit(n_inner_splits):
    """Select l1_ratio by minimum MSE, then its largest 1-SE alpha."""
    def select_one_se_index(cv_results):
        mean_mse = -np.asarray(cv_results["mean_test_score"])
        std_mse = np.asarray(cv_results["std_test_score"])
        alphas = np.asarray(
            [
                params["elastic_net__alpha"]
                for params in cv_results["params"]
            ]
        )
        l1_ratios = np.asarray(
            [
                params["elastic_net__l1_ratio"]
                for params in cv_results["params"]
            ]
        )

        index_min = int(np.argmin(mean_mse))
        selected_l1_ratio = l1_ratios[index_min]
        threshold = (
            mean_mse[index_min]
            + std_mse[index_min] / np.sqrt(n_inner_splits)
        )
        eligible = np.flatnonzero(
            np.isclose(l1_ratios, selected_l1_ratio)
            & (mean_mse <= threshold)
        )
        return int(eligible[np.argmax(alphas[eligible])])

    return select_one_se_index


def evaluate_elastic_net_one_se(
    regime,
    outer_folds,
    X_model,
    y,
    meta,
):
    """Nested interaction Elastic Net evaluation with 1-SE alpha."""
    model_name = "interaction_elastic_net"
    X_model = X_model.reset_index(drop=True)
    meta = meta.reset_index(drop=True)
    y = np.asarray(y, dtype=float)
    oof_prediction = np.full(len(y), np.nan, dtype=float)
    test_coverage = np.zeros(len(y), dtype=int)
    fold_rows = []

    for fold in outer_folds:
        train_idx = np.asarray(fold["train_idx"], dtype=int)
        test_idx = np.asarray(fold["test_idx"], dtype=int)
        X_train = X_model.iloc[train_idx]
        X_test = X_model.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        inner_splits, inner_details = make_repeated_regression_cv(
            X_train,
            y_train,
        )
        n_inner_splits = len(inner_splits)
        search = GridSearchCV(
            estimator=clone(elastic_net_pipeline),
            param_grid=ELASTIC_NET_PARAM_GRID,
            scoring="neg_mean_squared_error",
            cv=inner_splits,
            refit=make_elastic_net_one_se_refit(n_inner_splits),
            n_jobs=-1,
            return_train_score=False,
            error_score="raise",
        )
        search.fit(X_train, y_train)
        fold_prediction = np.asarray(
            search.predict(X_test),
            dtype=float,
        )

        cv_results = search.cv_results_
        mean_mse = -np.asarray(cv_results["mean_test_score"])
        std_mse = np.asarray(cv_results["std_test_score"])
        candidate_alphas = np.asarray(
            [
                params["elastic_net__alpha"]
                for params in cv_results["params"]
            ]
        )
        candidate_l1_ratios = np.asarray(
            [
                params["elastic_net__l1_ratio"]
                for params in cv_results["params"]
            ]
        )
        index_min = int(np.argmin(mean_mse))
        alpha_min = float(candidate_alphas[index_min])
        l1_ratio_min = float(candidate_l1_ratios[index_min])
        alpha_1se = float(
            search.best_params_["elastic_net__alpha"]
        )
        l1_ratio_selected = float(
            search.best_params_["elastic_net__l1_ratio"]
        )
        se_at_min = float(
            std_mse[index_min] / np.sqrt(n_inner_splits)
        )

        if not np.all(np.isfinite(fold_prediction)):
            raise ValueError(
                f"{regime}, {fold['fold_id']}: non-finite prediction"
            )
        if np.any(test_coverage[test_idx] != 0):
            raise ValueError(
                f"{regime}: an observation occurs in multiple test folds"
            )

        oof_prediction[test_idx] = fold_prediction
        test_coverage[test_idx] += 1
        fold_rows.append(
            {
                "model": model_name,
                "regime": regime,
                "fold_id": fold["fold_id"],
                "held_out_catalysts": ";".join(
                    fold["held_out_catalysts"]
                ),
                "held_out_substrates": ";".join(
                    fold["held_out_substrates"]
                ),
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_unused": fold["n_unused"],
                "alpha_min": alpha_min,
                "alpha_1se": alpha_1se,
                "alpha_selected": alpha_1se,
                "l1_ratio_min_mse": l1_ratio_min,
                "l1_ratio_selected": l1_ratio_selected,
                "alpha_choice": "1se_within_best_l1_ratio",
                "mean_inner_mse_min": float(mean_mse[index_min]),
                "se_inner_mse_at_min": se_at_min,
                "one_se_mse_threshold": float(
                    mean_mse[index_min] + se_at_min
                ),
                "r2": safe_r2(y_test, fold_prediction),
                "mae": mean_absolute_error(y_test, fold_prediction),
                "rmse": np.sqrt(
                    mean_squared_error(y_test, fold_prediction)
                ),
                **inner_details,
            }
        )

    if not np.all(test_coverage == 1):
        raise ValueError(
            f"{regime}: expected one prediction per observation; "
            f"coverage values were {np.unique(test_coverage)}"
        )

    prediction_table = meta.copy()
    prediction_table["model"] = model_name
    prediction_table["regime"] = regime
    prediction_table["y_true"] = y
    prediction_table["y_pred"] = oof_prediction
    prediction_table["residual"] = y - oof_prediction
    prediction_table["absolute_error"] = np.abs(
        prediction_table["residual"]
    )

    summary = {
        "model": model_name,
        "regime": regime,
        "n_observations": len(y),
        "n_outer_folds": len(outer_folds),
        "alpha_choice": "1se_within_best_l1_ratio",
        "r2": safe_r2(y, oof_prediction),
        "mae": mean_absolute_error(y, oof_prediction),
        "rmse": np.sqrt(mean_squared_error(y, oof_prediction)),
        "median_absolute_error": np.median(
            np.abs(y - oof_prediction)
        ),
    }
    return {
        "summary": summary,
        "predictions": prediction_table,
        "fold_metrics": pd.DataFrame(fold_rows),
    }


elastic_net_results = {}

for regime in validation_regimes:
    print(f"Evaluating interaction Elastic Net: {regime}")
    elastic_net_results[regime] = evaluate_elastic_net_one_se(
        regime=regime,
        outer_folds=split_registry[regime],
        X_model=X_lasso_interaction,
        y=y,
        meta=meta,
    )

elastic_net_summary = pd.DataFrame(
    [result["summary"] for result in elastic_net_results.values()]
).sort_values("regime").reset_index(drop=True)
elastic_net_predictions = pd.concat(
    [result["predictions"] for result in elastic_net_results.values()],
    ignore_index=True,
)
elastic_net_fold_metrics = pd.concat(
    [result["fold_metrics"] for result in elastic_net_results.values()],
    ignore_index=True,
)

display(elastic_net_summary.style.format(
    {
        "r2": "{:.3f}",
        "mae": "{:.3f}",
        "rmse": "{:.3f}",
        "median_absolute_error": "{:.3f}",
    }
))

In [ ]:
elastic_net_fold_metrics["alpha_1se_to_min_ratio"] = (
    elastic_net_fold_metrics["alpha_1se"]
    / elastic_net_fold_metrics["alpha_min"]
)

elastic_net_hyperparameter_summary = (
    elastic_net_fold_metrics.groupby(["model", "regime"], as_index=False)
    .agg(
        n_outer_folds=("fold_id", "nunique"),
        alpha_min_cv_median=("alpha_min", "median"),
        alpha_1se_min=("alpha_1se", "min"),
        alpha_1se_median=("alpha_1se", "median"),
        alpha_1se_max=("alpha_1se", "max"),
        l1_ratio_mode=(
            "l1_ratio_selected",
            lambda values: values.mode().iloc[0],
        ),
        median_alpha_1se_to_min_ratio=(
            "alpha_1se_to_min_ratio",
            "median",
        ),
    )
)

display(elastic_net_hyperparameter_summary.style.format(
    {
        "alpha_min_cv_median": "{:.5g}",
        "alpha_1se_min": "{:.5g}",
        "alpha_1se_median": "{:.5g}",
        "alpha_1se_max": "{:.5g}",
        "l1_ratio_mode": "{:.2f}",
        "median_alpha_1se_to_min_ratio": "{:.2f}",
    }
))

MODEL_DISPLAY_NAMES["interaction_elastic_net"] = (
    "Interaction Elastic Net"
)
EXTENDED_COMPARISON_MODELS = COMPARISON_MODELS + (
    "interaction_elastic_net",
)
extended_model_comparison_summary = pd.concat(
    [model_comparison_summary, elastic_net_summary],
    ignore_index=True,
)
extended_model_comparison_predictions = pd.concat(
    [model_comparison_predictions, elastic_net_predictions],
    ignore_index=True,
)

for metric in ["mae", "rmse", "r2"]:
    comparison = extended_model_comparison_summary.pivot(
        index="regime",
        columns="model",
        values=metric,
    ).reindex(validation_regimes)
    print(metric.upper())
    display(comparison.style.format("{:.3f}"))

extended_comparison_neutral_figure, _ = plot_model_comparison(
    prediction_table=extended_model_comparison_predictions,
    summary_table=extended_model_comparison_summary,
    model_names=EXTENDED_COMPARISON_MODELS,
)
extended_comparison_highlighted_figure, _ = plot_model_comparison(
    prediction_table=extended_model_comparison_predictions,
    summary_table=extended_model_comparison_summary,
    model_names=EXTENDED_COMPARISON_MODELS,
    highlight_colors=LASSO_HIGHLIGHT_COLORS,
)

ELASTIC_NET_RESULTS_DIR = Path("results/elastic_net")
ELASTIC_NET_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

elastic_net_summary.to_csv(
    ELASTIC_NET_RESULTS_DIR / "elastic_net_summary.csv",
    index=False,
)
elastic_net_predictions.to_csv(
    ELASTIC_NET_RESULTS_DIR / "elastic_net_predictions.csv",
    index=False,
)
elastic_net_fold_metrics.to_csv(
    ELASTIC_NET_RESULTS_DIR / "elastic_net_fold_metrics.csv",
    index=False,
)
elastic_net_hyperparameter_summary.to_csv(
    ELASTIC_NET_RESULTS_DIR / "elastic_net_hyperparameter_summary.csv",
    index=False,
)
extended_model_comparison_summary.to_csv(
    ELASTIC_NET_RESULTS_DIR / "model_comparison_summary.csv",
    index=False,
)
extended_model_comparison_predictions.to_csv(
    ELASTIC_NET_RESULTS_DIR / "model_comparison_predictions.csv",
    index=False,
)
extended_comparison_neutral_figure.savefig(
    ELASTIC_NET_RESULTS_DIR / "model_comparison_oof_neutral.png",
    dpi=300,
    bbox_inches="tight",
)
extended_comparison_highlighted_figure.savefig(
    ELASTIC_NET_RESULTS_DIR / "model_comparison_oof_A5_A6.png",
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved Elastic Net results to {ELASTIC_NET_RESULTS_DIR.resolve()}")